In [ ]:
# import torch
# import numpy as np
import dill

from full_models import *
# from curves import *
from interior_models import *
# from reparametrization_models import *
# from samplers import *
from plotting import *
# from losses import *

from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget

In [ ]:
from plotting import plot_four_poincare_models

%matplotlib widget

trained_models_path = 'pre-trained models/'

# The specific files you requested
model_files = [
    'ellipse_1.dill', 
    'gear_1.dill', 
    'star_2.dill', 
    'star_3.dill'
]

# Load them into a dictionary for plotting
models_to_plot = {}
for filename in model_files:
    with open(trained_models_path + filename, 'rb') as f:
        # Format the title nicely (e.g., 'ellipse_1.dill' -> 'Ellipse 1')
        title = filename.replace('.dill', '').replace('_', ' ').title()
        models_to_plot[title] = dill.load(f)

# Plot the 2x2 grid. 
# You can change the colormap to 'plasma', 'coolwarm', 'magma', etc.
plot_four_poincare_models(
    models_dict=models_to_plot, 
    colormap='Greys', 
    grid_size=1000,
    show_axes=False,
    show_grid=False,
    show_ticks=False,
    wspace=-0.1,  # Width spacing
    hspace=-0.1   # Height spacing
)

#### First we load the pre-trained model

In [ ]:
trained_models_path = 'pre-trained models/'
# model_name = 'cassini_1.dill'
# model_name = 'ellipse_1.dill'
# model_name = 'gear_1.dill'
model_name = 'star_1.dill'
# model_name = 'star_2.dill'
# model_name = 'star_3.dill'


with open(trained_models_path + model_name, 'rb') as f:
    trained_model = dill.load(f)

#### We plot the boundary curve

In [ ]:
curve = trained_model.curve # get_gear(R=2.0, amp=0.5, teeth=9, sharpness=3.0)
plot_curve_and_projection(curve)

#### We now show the initial approximation to our minimal surface

In [ ]:
# initial approximation model
interior_model = MLP(output_dim=3)

sampler = MixSampler(
    mix=1,
    bias=0.5,
    # target=(0,0),
    # sigma=0.1
)

first_model = MainModel(
    curve=curve,
    interior_model=interior_model,
    sampler=sampler,
    extension_method='stereographic',
)

#### MSE pre-training

In [ ]:
plot_error(
    first_model,
    vmax=1,
    grid_size=200)

#### Plot of the model pre-training

In [ ]:
plot_H3_surfaces(
    model_A=first_model, 
    model_B=trained_model, # not showed
    grid_size_A=500,
    grid_size_B=500,
    min_r_A = 0,
    max_r_A = 1,
    min_r_B = 0,
    max_r_B = 1,
    min_theta_A = 0,
    max_theta_A = 2*np.pi,
    min_theta_B = 0,
    max_theta_B = 2*np.pi,
    alpha_A = 0.5,
    alpha_B = 0)

#### Plot of the MSE post-training

In [7]:
plot_error(
    trained_model,
    vmax=1,
    grid_size=200)

#### Plot of the model post-training

In [8]:
plot_H3_surfaces(
    model_A=first_model, 
    model_B=trained_model,
    grid_size_A=500,
    grid_size_B=500,
    min_r_A = 0,
    max_r_A = 1,
    min_r_B = 0,
    max_r_B = 1,
    min_theta_A = 0,
    max_theta_A = 2*np.pi,
    min_theta_B = 0,
    max_theta_B = 2*np.pi,
    alpha_A = 0,
    alpha_B = 1)

#### Montecarlo distribution of the MSE

In [9]:
montecarlo_error(
    model = trained_model,
    loss_fcn = L2_squared,
    num_samples = 100,
    size_samples = 2000,
)